# 🎙️ وكيل Jais الصوتي — Jais Voice Agent

وكيل صوتي عربي متكامل: **يسمعك (ASR) ← يفكر ويجيب (Jais-3-7B) ← يتكلّم (TTS عربي)**.

### قبل التشغيل (مهم جداً)
1. **Runtime → Change runtime type → عُجلة GPU من نوع T4** (مجاني) → Save
2. ثم اضغط **Run all** (تشغيل الكل)
3. أول تشغيل ينزّل النموذج (~17GB) وقد يستغرق 10-20 دقيقة — انتظر حتى يظهر **الرابط العام** في الخلية الأخيرة
4. افتح الرابط ← اضغط 🎤 وتكلم بصوتك ← الوكيل يسمعك ويجاوب

> الجلسة تستمر طالما بقيت نوتة Colab مفتوحة. إذا انقطعت، اضغط Run all مرة أخرى.

In [ ]:
%pip -q install fastapi uvicorn pydantic transformers accelerate edge-tts faster-whisper huggingface_hub

In [ ]:
import os, json, subprocess
REPO = '/content/jais-voice-agent'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/911houssem/jais-voice-agent.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=False)

CONFIG = {
    'PORT': 8000,
    'ASR_SIZE': 'small',            # faster-whisper: tiny/base/small/medium/large-v3
    'ASR_DEVICE': 'cpu',
    'LLM_MODEL': 'inception42/Jais-3-7B',
    'TTS_VOICE': 'ar-SA-HamedNeural',
}
json.dump(CONFIG, open('/content/.jais_config.json', 'w'), ensure_ascii=False)
print('✅ الكود والإعدادات جاهزة:', CONFIG)

In [ ]:
import torch
if torch.cuda.is_available():
    print('✅ GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ لا يوجد GPU — اختر T4 من: Runtime → Change runtime type (النموذج لن يعمل بسرعة على المعالج)')

from huggingface_hub import snapshot_download
snapshot_download('inception42/Jais-3-7B')
print('✅ نموذج Jais-3-7B نُزّل وجاهز')

from faster_whisper import WhisperModel
WhisperModel('small', device='cpu', compute_type='int8')
print('✅ نموذج التعرّف على الكلام (ASR) جاهز')

In [ ]:
import sys, urllib.request
sys.path.insert(0, '/content/jais-voice-agent/kaggle')
import kaggle_app
kaggle_app.start_app(8000)
print('Health:', urllib.request.urlopen('http://127.0.0.1:8000/health').read().decode())

In [ ]:
import subprocess, re
subprocess.run(['wget', '-q',
                'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
                '-O', '/usr/local/bin/cloudflared'], check=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(120):
    line = proc.stdout.readline()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line or '')
    if m:
        url = m.group(0)
        break

if url:
    from IPython.display import display, HTML
    display(HTML(f'<h2>🔗 رابط الوكيل الصوتي (عام — افتحه وتكلم): <a href="{url}" target="_blank">{url}</a></h2>'))
else:
    print('تعذّر إنشاء الرابط — أعد تشغيل هذه الخلية')